In [1]:

import torch
import torch.nn as nn

def householder_orthogonalization(W):
    Q, _ = torch.linalg.qr(W)
    return Q


Strict Stable Koopman Operator

In [2]:

class StrictStableKoopmanOperator(nn.Module):
    def __init__(self, latent_dim, max_spectral_radius=0.99):
        super().__init__()

        self.latent_dim = latent_dim
        self.max_spectral_radius = max_spectral_radius

        self.U_raw = nn.Parameter(
            torch.randn(latent_dim, latent_dim)
        )

        self.V_raw = nn.Parameter(
            torch.randn(latent_dim, latent_dim)
        )

        self.S_unconstrained = nn.Parameter(
            torch.randn(latent_dim)
        )

    def forward(self, z):
        U = householder_orthogonalization(self.U_raw)
        V = householder_orthogonalization(self.V_raw)

        S = torch.sigmoid(self.S_unconstrained) * self.max_spectral_radius

        K = U @ torch.diag(S) @ V.T

        z_next = z @ K.T

        return z_next, K, S


In [ ]:

# Khởi tạo toán tử Koopman
torch.manual_seed(42)

latent_dim = 16
koopman = StrictStableKoopmanOperator(
    latent_dim=latent_dim,
    max_spectral_radius=0.99
)

# Tạo dữ liệu latent giả lập
z = torch.randn(4, latent_dim)

# Forward pass
z_next, K, S = koopman(z)

# Kiểm tra kích thước
print("Input z shape:", z.shape)
print("Output z_next shape:", z_next.shape)
print("Koopman matrix K shape:", K.shape)
print("Singular values S shape:", S.shape)

# Kiểm tra tính ổn định
singular_values = torch.linalg.svdvals(K) # mức khuếch đại lớn nhất của ma trận K
eigenvalues = torch.linalg.eigvals(K) # các giá trị riêng của ma trận K

spectral_norm = singular_values.max().item() # chuẩn toán tử bậc 2
spectral_radius = eigenvalues.abs().max().item() # bán kính phổ

print("\n=== Stability Check ===")
print("Max singular value:", spectral_norm)
print("Spectral radius:", spectral_radius)
print("Stable:", spectral_radius < 1.0)

# Kiểm tra độ dài vector latent
z_norm = torch.linalg.vector_norm(z, dim=1)
z_next_norm = torch.linalg.vector_norm(z_next, dim=1)

print("\n=== Latent Norm Check ===")
print("Before Koopman:", z_norm)
print("After Koopman:", z_next_norm)
print("Contraction satisfied:", torch.all(z_next_norm <= 0.99 * z_norm + 1e-6).item())


Input z shape: torch.Size([4, 16])
Output z_next shape: torch.Size([4, 16])
Koopman matrix K shape: torch.Size([16, 16])
Singular values S shape: torch.Size([16])

=== Stability Check ===
Max singular value: 0.810895562171936
Spectral radius: 0.4780125617980957
Stable: True

=== Latent Norm Check ===
Before Koopman: tensor([4.4333, 5.0287, 3.7499, 3.9687])
After Koopman: tensor([2.1793, 2.4723, 1.5100, 1.9987], grad_fn=<LinalgVectorNormBackward0>)
Contraction satisfied: True


PatchTSTBackbone

In [ ]:
# PatchTST Backbone
class PatchTSTBackbone(nn.Module):
    def __init__(self, input_dim, d_model, num_heads, dim_feedforward, num_layers):
        super().__init__()

        self.embedding = nn.Linear(input_dim, d_model)

        self.pos_encoding = nn.Parameter(torch.randn(1, d_model)) # phân biệt các thứ tự token trong chuỗi

        layers = nn.TransformerEncoderLayer(
            d_model, num_heads, dim_feedforward, batch_first=True
        )

        self.encoder = nn.TransformerEncoder(layers, num_layers)

    def forward(self, x):
        x = self.embedding(x) + self.pos_encoding
        return self.encoder(x)


Xây dựng DeepKoopFormer-PatchTST hoàn chỉnh  
PatchTST Backbone + Koopman Operator + Linear Decoder

In [7]:

# DeepKoopFormer - PatchTST Wrapper
class PatchTSTWrapper(nn.Module):
    def __init__(self, patch_length, horizon):
        super().__init__()

        self.backbone = PatchTSTBackbone(
            patch_length * 3, 16, 2, 64, 2
        )

        self.koopman = StrictStableKoopmanOperator(16)

        self.fc = nn.Linear(16, horizon * 3)

    def forward(self, x):
        x = x.reshape(x.shape[0], 1, -1)

        latent = self.backbone(x).mean(dim=1)

        latent_next, K, eigvals = self.koopman(latent)

        out = self.fc(latent_next)

        return out, latent, latent_next, K, eigvals



PositionalEncoding

In [9]:

import numpy as np

# Positional Encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(0, max_len).unsqueeze(1).float()

        div_term = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-np.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)].to(x.device)
    
    


In [ ]:

# Series Decomposition
class SeriesDecomposition(nn.Module):
    def __init__(self, kernel_size=25):
        super().__init__()

        self.moving_avg = nn.AvgPool1d(
            kernel_size, # batch size
            stride=1, # độ dài chuỗi
            padding=kernel_size // 2 # số đặc trưng
        )

    def forward(self, x):
        trend = self.moving_avg(x.transpose(1, 2)).transpose(1, 2)

        return trend, x - trend


Original: tensor([2., 4., 9., 5., 1.])
Trend: tensor([2., 5., 6., 5., 2.])
Seasonal: tensor([ 0., -1.,  3.,  0., -1.])
Reconstructed: tensor([2., 4., 9., 5., 1.])


AutoformerBackbone

In [12]:
# Autoformer Backbone
class AutoformerBackbone(nn.Module):
    def __init__(self, input_len, d_model):
        super().__init__()

        self.decomp = SeriesDecomposition(3)

        self.embed = nn.Linear(1, d_model)

        self.pos_encoding = PositionalEncoding(d_model)

        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model, 2, 32, batch_first=True
            ),
            2
        )

    def forward(self, x):
        trend, seasonal = self.decomp(x)

        seasonal = self.pos_encoding(
            self.embed(seasonal)
        )

        latent = self.encoder(seasonal).mean(dim=1)

        return latent


AutoformerWrapper

In [15]:
# DeepKoopFormer - Autoformer Wrapper
class AutoformerWrapper(nn.Module):
    def __init__(self, patch_length, horizon):
        super().__init__()

        self.backbone = AutoformerBackbone(
            patch_length * 3, 16
        )

        self.koopman = StrictStableKoopmanOperator(16)

        self.fc = nn.Linear(16, horizon * 3)

    def forward(self, x):
        x = x.reshape(x.shape[0], -1, 1)

        latent = self.backbone(x)

        latent_next, K, eigvals = self.koopman(latent)

        out = self.fc(latent_next)

        return out, latent, latent_next, K, eigvals


InformerSparse

In [14]:
# Informer Backbone
class InformerSparse(nn.Module):
    def __init__(self, input_dim, d_model):
        super().__init__()

        self.embed = nn.Linear(input_dim, d_model)

        self.pos_encoding = PositionalEncoding(d_model)

        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model, 2, 32, batch_first=True
            ),
            2
        )

    def forward(self, x):
        x = self.pos_encoding(self.embed(x))

        return self.encoder(x).mean(dim=1)


# Test Informer Backbone
informer = InformerSparse(
    input_dim=3,
    d_model=16
)

dummy_x = torch.randn(2, 150, 3)

output = informer(dummy_x)

print("Input shape:", dummy_x.shape)
print("Latent shape:", output.shape)


Input shape: torch.Size([2, 150, 3])
Latent shape: torch.Size([2, 16])


In [16]:
# DeepKoopFormer - Informer Wrapper
class InformerWrapper(nn.Module):
    def __init__(self, patch_length, horizon):
        super().__init__()

        self.backbone = InformerSparse(3, 16)

        self.koopman = StrictStableKoopmanOperator(16)

        self.fc = nn.Linear(16, horizon * 3)

    def forward(self, x):
        latent = self.backbone(x)

        latent_next, K, eigvals = self.koopman(latent)

        out = self.fc(latent_next)

        return out, latent, latent_next, K, eigvals
